# Module 8.1: Sparse Mixture of Experts (MoE)

Welcome to the bleeding edge of Transformer Architecture!

By now, you have built the entire "Dense" Transformer model. Every single token inputted into the model is multiplied against every single mathematical weight in the network. 

In this notebook, we look at the ultimate architectural trick designed to scale models to trillions of parameters *without* destroying latency speeds: **Mixture of Experts**.

## 1. WHAT is Mixture of Experts (MoE)?

In a standard Transformer, the Feed-Forward Network (FFN) located inside every Encoder/Decoder block is a massive block of parameters. 

**MoE** breaks this single massive FFN into multiple smaller FFNs called "Experts". Instead of a token passing through one giant matrix, a special "Router" network looks at the token and sends it to only the top $K$ experts (usually 2)! 

### \ud83c\udfd7\ufe0f The Construction Analogy
Imagine you are building a house (Predicting a token).
- **Dense Network**: You have one "Jack-of-all-Trades" worker. Every time a task arises (Plumbing, Electricity, Painting), this one poor guy has to do it. If the house gets complex, you have to replace him with an incredibly expensive, slow super-worker.
 
- **MoE**: You hire 8 specialized workers (The Experts) and 1 Foreman (The Router). When a "pipe" task arrives, the Foreman instantly points to the Plumber. The other 7 workers sit idle. It is immensely fast, and strictly specialized!

## 2. WHY do we use MoE?

**Decoupling Parameter Count from Compute.**

If you want a smarter model, you usually make it wider and deeper. But an 8x bigger model takes 8x longer to run. 

With MoE like *Mixtral 8x7B*, the model has 47 Billion parameters in memory (VRAM). BUT, because the Router only activates 2 out of the 8 experts per token, each token only "sees" 13 Billion parameters of math. You get the intelligence capacity of a 47B model, running at the blazing fast inference speed of a 13B model! This is called **Sparse Scaling**.

## 3. HOW does MoE work? (The Architecture)

The Attention mechanisms work exactly the same as always. 
The magic happens when the data reaches the Feed-Forward Layer.

1. The Input Token hits the **Router Network** (a simple Linear layer resolving to probabilities).
2. The Router calculates a score for how much it "likes" each of the 8 experts.
3. We pick the `Top-K` scores (e.g., Top 2).
4. The token is explicitly routed to those two expert FFNs.
5. Their processed outputs are mathematically added together, weighted by the Router's confidence score.

```mermaid
graph TD
    Token[Input Token\nDim: 4096] --> Router[Router Network\nOutputs 8 Scores]
    
    Router -->|Top 1: Score 0.7| E1[Expert 1\nFeed-Forward]
    Router -.->|Not Chosen: Score 0.05| E2[Expert 2]
    Router -.->|Not Chosen: Score 0.01| E3[Expert 3]
    Router -->|Top 2: Score 0.2| E4[Expert 4\nFeed-Forward]
    
    E1 -->|Multiply by 0.7| Sum{+}
    E4 -->|Multiply by 0.2| Sum
    
    Sum --> Out[Output Prediction]
```

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class Expert(nn.Module):
    """
    A standard Feed-Forward Network. Same as we built in Module 3!
    """
    def __init__(self, d_model):
        super().__init__()
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_model * 4),
            nn.ReLU(),
            nn.Linear(d_model * 4, d_model)
        )
    def forward(self, x):
        return self.ffn(x)

class SparseMixtureOfExperts(nn.Module):
    def __init__(self, d_model=128, num_experts=8, top_k=2):
        super().__init__()
        self.num_experts = num_experts
        self.top_k = top_k
        
        # The Foreman : Looks at the token (d_model) and outputs (num_experts) scores
        self.router = nn.Linear(d_model, num_experts, bias=False)
        
        # The Workers: A list of independent FFNs.
        self.experts = nn.ModuleList([Expert(d_model) for _ in range(num_experts)])

    def forward(self, x):
       # x shape: (Batch, Sequence, d_model)
       
       # 1. The Router "grades" the token for all 8 experts
       # Shape: (Batch, Sequence, 8)
       router_logits = self.router(x)
       
       # Get probabilities so they sum to 1.0
       routing_probs = F.softmax(router_logits, dim=-1)
       
       # 2. Pick the Top K (Best 2 Experts)
       # top_k_probs Shape: (Batch, Sq, 2)
       # top_k_indices Shape: (Batch, Sq, 2) <- Contains the ID of the chosen experts (0-7)
       top_k_probs, top_k_indices = torch.topk(routing_probs, self.top_k, dim=-1)
       
       # We re-normalize the probabilities of JUST the 2 chosen experts 
       # so they sum to 1.0 for clean math.
       top_k_probs = top_k_probs / top_k_probs.sum(dim=-1, keepdim=True)
       
       final_output = torch.zeros_like(x)
       
       # 3. Mathematically route the data!
       # (In PyTorch we simulate this cleanly. In reality, extreme CUDA tricks are used to batch them physically together)
       for i, expert in enumerate(self.experts):
           # Find where THIS expert 'i' was picked in the top 2 indices
           expert_mask = (top_k_indices == i)
           
           if expert_mask.any():
               # Calculate output of the expert
               expert_out = expert(x)
               
               # We multiply the expert's output by the confidence probability 
               # the router gave it, and add it to the final tracking output.
               for k in range(self.top_k):
                   mask_for_this_k = expert_mask[..., k]
                   # Injecting the routed FFN outputs recursively
                   final_output[mask_for_this_k] += expert_out[mask_for_this_k] * top_k_probs[mask_for_this_k, k].unsqueeze(-1)

       return final_output

# Look at the logic in action!
moe_layer = SparseMixtureOfExperts(d_model=16, num_experts=8, top_k=2)
dummy_tokens = torch.randn(1, 4, 16) # 1 sentence, 4 words.

print("Pushing tokens through the Mixture of Experts!")
out = moe_layer(dummy_tokens)
print(f"Final Output Shape: {out.shape} -> Look! It perfectly matched the Dense input!\n\nSparse Scaling Achieved.")

## Summary

By adding a `Softmax Router` and turning our Dense FFN layer into a list of Experts, we achieved the holy grail of Machine Learning: increasing intelligence capacity massively, without a massive hit to computing speed delays.

The final trick to MoE training is the **Load Balancing Loss**. If unchecked, the Router gets "lazy" and just sends every token to Expert 1 (effectively killing the other 7 workers and destroying the parameter scaling). AI Engineers inject a mathematical penalty into the overall Loss function to explicitly punish the Router if the distribution across all 8 experts is uneven!

You have officially mastered the advanced structural layout of the largest Deep Learning pipelines in the modern era.